# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, processing, and exploring a FAIR^2-compliant dataset using the `mlcroissant` library, focusing on referencing all elements by their `@id` for maximum reproducibility.

### Dataset Source
The dataset source is defined by a Croissant schema:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and records from the dataset. All entity lookups and data extraction will use Croissant `@id` fields as required.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
ds = mlc.Dataset(croissant_url)
meta = ds.metadata

print("Dataset Title:", meta.name)
print("Description:\n", meta.description)

## 2. Data Overview
Print all available `@id`s for record sets, the associated fields, and columns. Reference entities by `@id`.

In [ ]:
# List all record sets and their @id, fields, and columns
print("Available Record Sets (by @id):")
for record_set in meta.record_sets:
    print(f"- Record Set @id: {record_set['@id']}")
    print(f"  Name: {record_set['name']}")
    # Fields in the record set
    if 'fields' in record_set:
        print("  Fields (by @id):")
        for field in record_set['fields']:
            print(f"    - {field['@id']}: {field['name']}")
            # Columns attached to this field
            if 'columns' in field:
                print("      Columns (by @id):")
                for col in field['columns']:
                    print(f"        - {col['@id']}: {col.get('name', '<no name>')}")
    print("")

# For demonstration, iterate a few records from the first record set
if meta.record_sets:
    example_rs_id = meta.record_sets[0]['@id']
    print(f"\nFew sample records from record set @id '{example_rs_id}':")
    for i, record in enumerate(ds.records(record_set=example_rs_id)):
        pprint(record)
        if i >= 2:
            break

## 3. Data Extraction
Load all record sets into Pandas DataFrames for analysis, using their `@id`. You can then select a specific record set for further analysis.

In [ ]:
# Collect all record set @id for extraction
record_set_ids = [rs['@id'] for rs in meta.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records from record set @id: {rs_id}")
    try:
        records = list(ds.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f" - Columns: {df.columns.tolist()}")
        print(f" - Number of records: {len(df)}\n")
    except Exception as e:
        print(f"Failed to load records for {rs_id} due to: {e}")

# For further analysis, select the main record set
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Sample from main record set (@id {main_rs_id}):")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's choose a numeric field (`@id`) from our main record set for basic filtering and normalization, referencing all data points by their `@id`.

> **Tip:** Use the output above to select appropriate `@id`s for fields/columns for your own use-cases.

In [ ]:
# Define the @id for numeric field to analyze (adjust as needed for your dataset)
# For demonstration, attempt to pick first float/integer field
df = dataframes[main_rs_id]
numeric_col_id = None
print("Columns available:", df.columns.tolist())

for col in df.columns:
    # Try detecting a numeric column
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_col_id = col
        break
if numeric_col_id is None:
    raise RuntimeError("No numeric field found! Please adjust to a numeric field using its @id.")
print(f"Using numeric field (by @id): {numeric_col_id}\n")

# Filtering
threshold = df[numeric_col_id].mean() if pd.notnull(df[numeric_col_id].mean()) else 10
filtered_df = df[df[numeric_col_id] > threshold]
print(f"Filtered records with {numeric_col_id} > {threshold} (ref by @id): {len(filtered_df)} records")
display(filtered_df.head())

# Normalization (Z-score)
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_col_id}_normalized"] = (
    filtered_df[numeric_col_id] - filtered_df[numeric_col_id].mean()
) / filtered_df[numeric_col_id].std()
print(f"Sample normalized values for {numeric_col_id} (all references by @id):")
display(filtered_df[[numeric_col_id, f"{numeric_col_id}_normalized"]].head())

# Try grouping by a categorical field (pick first object/string type field that is not the numeric field)
group_field = None
for col in df.columns:
    # Exclude numeric_col_id and ignore fields with > 20 unique values (to avoid many tiny groups)
    if col != numeric_col_id and df[col].dtype == object and df[col].nunique() > 1 and df[col].nunique() < 20:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_col_id].mean().reset_index()
    print(f"Grouped mean {numeric_col_id} by {group_field} (by @id):")
    display(grouped_df)
else:
    print("No suitable categorical field for grouping found.")

## 5. Visualization
Visualize the distribution of the selected numeric field (`@id`: `{numeric_col_id}`) and relationship with the group field if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_col_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_col_id} (by @id)")
plt.xlabel(numeric_col_id)
plt.ylabel("Count")
plt.show()

# If grouping available, show boxplot
if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=group_field, y=numeric_col_id)
    plt.title(f"{numeric_col_id} by {group_field} (ref by @id)")
    plt.xlabel(group_field)
    plt.ylabel(numeric_col_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated loading and analyzing a FAIR^2-compliant dataset using the `mlcroissant` library, always referencing Croissant entities by their `@id`. We explored record sets, inspected field and column identifiers, extracted data into DataFrames, performed basic data transformations and filtering, and created simple visualizations. For deeper scientific analysis, consult the Croissant schema for domain-specific metadata (`@id` references of fields/columns for biological interpretation).

**Tips:**
- Always reference data elements in FAIR^2 and Croissant workflows by their `@id` for maximum interpretability and script portability.
- For further analysis, consult dataset documentation or domain experts to select correct field `@id`s.